# 01 — Inspeksi data: foto contoh, orientasi, anotasi

Foto contoh berlisensi dari `samples/catalog.json` (katalog yang sama dengan `GET /api/v1/samples`). Setiap
berkas dicek sha256-nya. Gambar ditampilkan **setelah orientasi EXIF** lewat `app.imaging` — sama dengan editor
web dan input model. Anotasi manusia (Pascal VOC) hanya untuk evaluasi di notebook 03.

Kernel: **Python (wallacea-ml)**. Tidak butuh model/bobot.

In [ ]:
%matplotlib inline
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display

import wallacea_ml
from wallacea_ml.prepare import ML_ROOT
print("wallacea_ml", wallacea_ml.__version__, "| ML_ROOT =", ML_ROOT)


def _df(rows):
    """Tabel: pandas bila terpasang (opsional), selain itu list apa adanya."""
    try:
        import pandas as pd
        return pd.DataFrame(rows)
    except ImportError:
        return rows


## 1. Katalog: sumber, lisensi, checksum

In [ ]:
from wallacea_ml.samples import load_samples, samples_dir
samples = load_samples()          # gagal keras bila sha256 tidak cocok dengan katalog
print("folder:", samples_dir())
display(_df([{"id": s.id, "berkas": s.path.name, "sha256": s.sha256[:16] + "…", "lisensi": s.license_name,
              "anotasi": s.annotation_labels} for s in samples]))
print("Sumber:", samples[0].source_url)
print("Atribusi:", samples[0].attribution)
print("Citra:", samples[0].imagery)

## 2. Dimensi mentah vs terorientasi, EXIF, kecerahan & ketajaman relatif

In [ ]:
from wallacea_ml import backend
from wallacea_ml.inspection import photo_folder_stats
rows = []
for s in samples:
    img = backend.load_oriented(s.path)
    raw = backend.raw_size(s.path)
    rows.append({"id": s.id, "mentah": f"{raw[0]}×{raw[1]}", "terorientasi": f"{img.width}×{img.height}",
                 "exif_orientation": backend.exif_orientation(s.path), "mode": img.mode, "byte": s.path.stat().st_size})
display(_df(rows))
display(_df([{k: r[k] for k in ("image", "brightness", "sharpness_lapvar_1024")}
             for r in photo_folder_stats(samples[0].path.parent)]))

## 3. Foto + anotasi manusia (hijau). Label asli dipetakan eksplisit.

In [ ]:
from wallacea_ml.viz import draw_boxes
from wallacea_ml.voc import load_voc
# OSBS & YELL memakai label "Tree"; SOAP_061 memakai "Alive"/"Dead". Semuanya tajuk pohon → plant (eksplisit).
VOC_LABELS = {"Tree": "plant", "Alive": "plant", "Dead": "plant"}
fig, axes = plt.subplots(1, len(samples), figsize=(6 * len(samples), 6))
size_rows = []
for ax, s in zip(axes, samples):
    img = backend.load_oriented(s.path)
    voc = load_voc(s.annotations, VOC_LABELS)
    assert (voc.width, voc.height) == img.size, "dimensi anotasi ≠ gambar terorientasi"
    ax.imshow(draw_boxes(img, voc.boxes)); ax.set_title(f"{s.id}: {len(voc.boxes)} kotak manusia"); ax.axis("off")
    w_px = sorted(b[2] * img.width for _, b in voc.boxes)
    size_rows.append({"id": s.id, "kotak": len(voc.boxes), "label_asli": voc.raw_labels,
                      "lebar_px_min": round(w_px[0], 1), "lebar_px_median": round(w_px[len(w_px) // 2], 1),
                      "lebar_px_max": round(w_px[-1], 1), "dilewati_clip": voc.skipped})
plt.tight_layout(); plt.show()
display(_df(size_rows))

## 4. Foto K919 (bila ada)

Salin foto ASLI K919 MAX ke `ml/data/k919/raw/` (tidak masuk Git). Sel ini hanya statistik — tanpa label,
tanpa klaim akurasi.

In [ ]:
K919 = ML_ROOT / "data" / "k919" / "raw"
k_rows = photo_folder_stats(K919) if K919.is_dir() else []
if k_rows:
    display(_df(k_rows))
    print("orientasi EXIF:", Counter(r["exif_orientation"] for r in k_rows))
else:
    print("Belum ada foto K919 di", K919, "— inspeksi hanya pada foto contoh NEON.")